# E01 — Raw spectral data audit
SeaweedGuard | Dryad DOI: 10.5061/dryad.f1vhhmh59

Inventory the original nested ZIP archives, identify readable numeric spectra, flag duplicates and missing values. No class or independent sample labels are inferred from filenames. Binary formats are inventoried, not silently converted. No modelling in this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import io, zipfile, hashlib, csv, json, re
import numpy as np
import pandas as pd
from IPython.display import display
ARCHIVE = Path('/content/drive/MyDrive/Thermodynamics/doi_10_5061_dryad_f1vhhmh59__v20241126.zip')
OUT = Path('/content/drive/MyDrive/SeaweedGuard/01_Data_Audit')
assert ARCHIVE.is_file(), f'Archive not found: {ARCHIVE}'
OUT.mkdir(parents=True, exist_ok=True)
def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()
print('Archive bytes:', ARCHIVE.stat().st_size)
print('SHA256:', sha256_file(ARCHIVE))

## 1. Archive inventory
Read in memory, without extracting untrusted filenames. Bound individual member sizes and recursion. Every skipped file is recorded.

In [ ]:
MAX_BYTES = 100 * 1024**2
MAX_TOTAL = 600 * 1024**2
MAX_DEPTH = 4
inventory, candidates = [], []
budget = 0
def walk(z, prefix='', depth=0):
    global budget
    for info in z.infolist():
        if info.is_dir():
            continue
        name = prefix + info.filename
        ext = Path(info.filename).suffix.lower()
        row = dict(path=name, extension=ext, bytes=info.file_size, status='inventoried', sha256=None)
        inventory.append(row)
        if depth > MAX_DEPTH or info.file_size > MAX_BYTES or budget + info.file_size > MAX_TOTAL:
            row['status'] = 'skipped_size_or_depth_limit'
            continue
        try:
            data = z.read(info)
            budget += len(data)
            row['sha256'] = hashlib.sha256(data).hexdigest()
            if ext == '.zip':
                if depth >= MAX_DEPTH:
                    row['status'] = 'skipped_depth_limit'
                else:
                    with zipfile.ZipFile(io.BytesIO(data)) as nested:
                        row['status'] = 'nested_archive'
                        walk(nested, name+'::', depth+1)
            elif ext in {'.csv','.txt','.tsv','.asc','.dat','.xy','.dpt'}:
                candidates.append((name, data))
                row['status'] = 'text_candidate'
            else:
                row['status'] = 'format_requires_review'
        except Exception as e:
            row['status'] = 'read_error: '+str(e)[:200]
with zipfile.ZipFile(ARCHIVE) as z:
    walk(z)
inv = pd.DataFrame(inventory)
inv.to_csv(OUT/'archive_inventory.csv', index=False)
display(inv.groupby(['extension','status'],dropna=False).size().rename('files').reset_index())
display(inv.head(20))

## 2. Conservative numeric spectrum parsing
Only two-column numeric tables with at least 20 finite paired observations and a predominantly monotonic first column qualify as provisional spectra. Headers, metadata and unsupported formats require review. A qualifying file is not automatically a validated FTIR spectrum.

In [ ]:
def parse_candidate(data):
    for encoding in ('utf-8-sig','latin-1'):
        try:
            content = data.decode(encoding)
            break
        except UnicodeDecodeError:
            continue
    lines = content.splitlines()
    for sep in (',',';','\t',r'\s+'):
        pairs = []
        malformed = 0
        for line in lines:
            line = line.strip()
            if not line or line.startswith(('#','//')):
                continue
            fields = re.split(sep,line) if sep == r'\s+' else line.split(sep)
            if len(fields) != 2:
                malformed += 1
                continue
            try:
                pairs.append([float(x.strip().strip('"')) for x in fields])
            except ValueError:
                malformed += 1
        if len(pairs) < 20:
            continue
        a = np.asarray(pairs,dtype=float)
        finite = np.isfinite(a).all(axis=1)
        x = a[finite,0]
        if len(x)<20:
            continue
        dx = np.diff(x)
        monotonic = max(np.mean(dx>0),np.mean(dx<0))
        if monotonic < .95:
            continue
        return a, malformed, sep
    return None, None, None

rows = []
for name, data in candidates:
    a, malformed, sep = parse_candidate(data)
    row = {'path':name,'parse_status':'unparsed_review_required'}
    if a is not None:
        finite = np.isfinite(a).all(axis=1)
        b = a[finite]
        x,y = b.T
        order = np.argsort(x,kind='stable')
        row.update(parse_status='provisional_numeric_spectrum',n_rows=len(a),
                   nonfinite_rows=int((~finite).sum()),unparsed_lines=malformed,
                   x_min=float(x.min()),x_max=float(x.max()),
                   median_abs_step=float(np.median(np.abs(np.diff(x)))),
                   duplicate_x=int(len(x)-len(np.unique(x))),
                   y_min=float(y.min()),y_max=float(y.max()),
                   numeric_sha256=hashlib.sha256(np.asarray(b[order],dtype='<f8').tobytes()).hexdigest(),
                   delimiter=sep)
    rows.append(row)
audit = pd.DataFrame(rows,columns=['path','parse_status','n_rows','nonfinite_rows','unparsed_lines','x_min','x_max','median_abs_step','duplicate_x','y_min','y_max','numeric_sha256','delimiter'])
audit.to_csv(OUT/'provisional_spectral_audit.csv',index=False)
display(audit['parse_status'].value_counts())
display(audit.head(20))

## 3. Duplicates and manual metadata reconciliation
Exact hashes detect identical content, not all correlated measurements. Repeated measurements of one specimen must stay together during validation. Confirm the meaning and units of each axis and the material, specimen, treatment, instrument and source-library provenance using original documentation.

In [ ]:
dups = inv[inv.sha256.notna() & inv.sha256.duplicated(keep=False)].sort_values('sha256')
dups.to_csv(OUT/'exact_file_duplicates.csv',index=False)
numdups = audit[audit.numeric_sha256.notna() & audit.numeric_sha256.duplicated(keep=False)]
numdups.to_csv(OUT/'exact_numeric_duplicates.csv',index=False)
template = inv.loc[~inv.status.eq('nested_archive'),['path']].copy()
for col in ['material_label','specimen_id','replicate_id','fouling_condition','instrument','axis_units','signal_units','reference_or_test','evidence_source','review_status']:
    template[col] = ''
template_path = OUT/'metadata_review_template.csv'
if not template_path.exists():
    template.to_csv(template_path,index=False)
summary = {'archive_sha256':sha256_file(ARCHIVE),'inventory_members':len(inv),
           'text_candidates':len(candidates),'provisional_spectra':int(audit.parse_status.eq('provisional_numeric_spectrum').sum()),
           'duplicate_file_rows':len(dups),'duplicate_numeric_rows':len(numdups),
           'independent_specimen_count':None,'verified_material_class_count':None}
(OUT/'audit_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('Outputs:',OUT)

## Exit criteria
E01 is not complete until unsupported formats are resolved, labels and independent sample groups are verified, duplicates are reconciled, and reference/test overlap is checked. Do not train from the provisional inventory alone.